[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/12_conv_and_vision_blocks.ipynb)

# 12. Convolution and vision blocks — faithful block anatomy

기본 convolution variant는 그대로 두되, 이전 ResNet bottleneck에서 빠졌던 **BatchNorm과 residual-add 뒤 activation**, ConvNeXt에서 빠졌던 **layer scale**을 보강했다.

목적은 kernel 종류만 보는 것이 아니라 실제 vision backbone block에서 convolution, normalization, activation, residual이 어떤 순서로 연결되는지 보는 것이다.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)


## 1. Standard, grouped, and depthwise-separable convolution

standard convolution은 모든 input channel을 함께 섞는다. grouped convolution은 channel group별로 독립 convolution을 수행하고, depthwise convolution은 `groups=in_channels`로 channel마다 spatial filter를 따로 적용한다. MobileNet의 depthwise-separable convolution은 뒤에 1×1 pointwise convolution을 붙여 channel mixing을 복구한다.


In [ ]:
x = torch.randn(1, 4, 16, 16, device=device)

standard_conv = nn.Conv2d(
    4, 8,
    kernel_size=3,
    padding=1,
).to(device)
grouped_conv = nn.Conv2d(
    4, 8,
    kernel_size=3,
    padding=1,
    groups=2,
).to(device)
depthwise_conv = nn.Conv2d(
    4, 4,
    kernel_size=3,
    padding=1,
    groups=4,
).to(device)
pointwise_conv = nn.Conv2d(
    4, 8,
    kernel_size=1,
).to(device)

standard_output = standard_conv(x)
grouped_output = grouped_conv(x)
depthwise_separable_output = pointwise_conv(depthwise_conv(x))

print("standard:", standard_output.shape)
print("grouped:", grouped_output.shape)
print("depthwise + pointwise:", depthwise_separable_output.shape)


## 2. Dilated convolution

dilation은 kernel sample 위치 사이 간격을 벌려 parameter 수를 늘리지 않고 receptive field를 키운다.


In [ ]:
dilated_conv = nn.Conv2d(
    4, 4,
    kernel_size=3,
    padding=2,
    dilation=2,
).to(device)

dilated_output = dilated_conv(x)
print("dilated output:", dilated_output.shape)


## 3. Original ResNet bottleneck anatomy

ResNet bottleneck은 단순 `1×1 → 3×3 → 1×1` convolution 세 개가 아니다. original post-activation bottleneck에서는 각 convolution 뒤 BatchNorm을 사용하고, 마지막 BN 뒤 shortcut을 더한 후 ReLU를 적용한다. channel/spatial shape가 바뀌면 shortcut에도 projection을 둔다.


In [ ]:
class ResNetBottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_channels=64, bottleneck_channels=16, stride=1):
        super().__init__()

        out_channels = bottleneck_channels * self.expansion

        self.conv1 = nn.Conv2d(
            in_channels,
            bottleneck_channels,
            kernel_size=1,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(bottleneck_channels)

        self.conv2 = nn.Conv2d(
            bottleneck_channels,
            bottleneck_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(bottleneck_channels)

        self.conv3 = nn.Conv2d(
            bottleneck_channels,
            out_channels,
            kernel_size=1,
            bias=False,
        )
        self.bn3 = nn.BatchNorm2d(out_channels)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)

        hidden = F.relu(self.bn1(self.conv1(x)))
        hidden = F.relu(self.bn2(self.conv2(hidden)))
        hidden = self.bn3(self.conv3(hidden))

        output = F.relu(hidden + identity)
        return output


resnet_input = torch.randn(2, 64, 16, 16, device=device)
resnet_block = ResNetBottleneck().to(device)
resnet_output = resnet_block(resnet_input)

print("ResNet bottleneck output:", resnet_output.shape)


## 4. ConvNeXt block: large depthwise conv + LayerNorm + inverted bottleneck + layer scale

ConvNeXt block은 7×7 depthwise convolution 뒤 channel-last LayerNorm과 `C → 4C → C` MLP를 사용한다. 원 논문의 block에는 작은 learnable layer-scale `gamma`도 있어 residual branch의 초기 크기를 제어한다. stochastic depth는 여기서는 생략한다.


In [ ]:
class ConvNeXtBlock(nn.Module):
    def __init__(self, channels=32, layer_scale_init=1e-6):
        super().__init__()

        self.depthwise = nn.Conv2d(
            channels,
            channels,
            kernel_size=7,
            padding=3,
            groups=channels,
        )
        self.norm = nn.LayerNorm(channels, eps=1e-6)
        self.pointwise1 = nn.Linear(channels, 4 * channels)
        self.pointwise2 = nn.Linear(4 * channels, channels)
        self.gamma = nn.Parameter(
            layer_scale_init * torch.ones(channels)
        )

    def forward(self, x):
        residual = x

        hidden = self.depthwise(x)
        hidden = hidden.permute(0, 2, 3, 1)
        hidden = self.norm(hidden)
        hidden = self.pointwise1(hidden)
        hidden = F.gelu(hidden)
        hidden = self.pointwise2(hidden)
        hidden = self.gamma * hidden
        hidden = hidden.permute(0, 3, 1, 2)

        return residual + hidden


convnext_input = torch.randn(2, 32, 16, 16, device=device)
convnext_block = ConvNeXtBlock().to(device)
convnext_output = convnext_block(convnext_input)

print("ConvNeXt output:", convnext_output.shape)
print("initial layer-scale mean:", convnext_block.gamma.mean().item())


## References and provenance

**ResNet** — He et al., *Deep Residual Learning for Image Recognition*. original bottleneck의 Conv-BN-ReLU ordering, projection shortcut, residual add 뒤 ReLU를 반영했다. pre-activation ResNet은 다른 ordering이다.

**MobileNet** — Howard et al. depthwise spatial convolution + pointwise channel mixing을 반영했다.

**ConvNeXt** — Liu et al., *A ConvNet for the 2020s*. 7×7 depthwise conv, channel-last LayerNorm, 4× inverted bottleneck MLP, GELU, layer scale, residual 구조를 반영했다. stochastic depth는 tiny example에서 생략했다.
